In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from src.data_info import *
from src.cleaning import *

df = load_data("data/vgsales.csv")

print("Before cleaning:")
size_data(df)
display(is_missing(df))

repartition(df)

df = clean_data(df)

print("\nAfter cleaning:")
size_data(df)

Before cleaning:
Number of line : 16598
Nombre of column : 11


Year            271
Publisher        58
Rank              0
Platform          0
Name              0
Genre             0
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

EU_Sales
<1 000            34.522232
1 000-10 000       9.013134
10 000-100 000    30.750693
100 000-1M        22.882275
1M-20M             2.825642
20M<               0.006025
Name: proportion, dtype: float64

After cleaning:
Number of line : 10667
Nombre of column : 11


In [3]:
from src.preparation import *
X_train, X_test, y_train, y_test = split_data(df)
preprocessor = create_preprocessor(X_train)

Training data: (8533, 8)
Test data: (2134, 8)
Numeric values: ['Year', 'NA_Sales', 'JP_Sales', 'Other_Sales']
Categorical values: ['Name', 'Platform', 'Genre', 'Publisher']


In [5]:
from src.model import *

result_random_forest = train_random_forest(preprocessor, X_train, y_train)

# Récupération du meilleur modèle
best_rf = result_random_forest.best_estimator_

# Prédictions sur les données de test
rf_train_predictions = best_rf.predict(X_train)
rf_test_predictions = best_rf.predict(X_test)

Fitting 5 folds for each of 150 candidates, totalling 750 fits
Best parameters RF: {'model__n_estimators': 500, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': 0.7, 'model__max_depth': 30, 'model__bootstrap': False}
RMSE CV RF: 0.20758506946913644


In [6]:
from src.model import *

# Entraînement du modèle HGB
result_hgb = train_hist_gradient_boosting(preprocessor,X_train,y_train)

# Récupération du meilleur modèle
best_hgb = result_hgb.best_estimator_

# Prédictions sur les données de test
hgb_train_predictions = best_hgb.predict(X_train)
hgb_test_predictions = best_hgb.predict(X_test)

Fitting 5 folds for each of 150 candidates, totalling 750 fits
Meilleurs paramètres HGB : {'model__min_samples_leaf': 10, 'model__max_leaf_nodes': 15, 'model__max_iter': 500, 'model__max_depth': 10, 'model__learning_rate': 0.15, 'model__l2_regularization': 50}
RMSE CV HGB : 0.20544674646325273


In [7]:
from src.evaluation import evaluate_model

# Évaluation des modèles
rf_train = evaluate_model(y_train, rf_train_predictions, "Random Forest - Train")
rf_test = evaluate_model(y_test, rf_test_predictions, "Random Forest - Test")

hgb_train = evaluate_model(y_train, hgb_train_predictions, "HistGradientBoosting - Train")
hgb_test = evaluate_model(y_test, hgb_test_predictions, "HistGradientBoosting - Test")

results = pd.DataFrame([
    rf_train,
    rf_test,
    hgb_train,
    hgb_test
])

display(results)


Random Forest - Train
MAE: 0.015284908706965106
RMSE: 0.07159980453831358
R²: 0.9817742659535924

Random Forest - Test
MAE: 0.06262864862532358
RMSE: 0.46752265963650497
R²: 0.7151089897343599

HistGradientBoosting - Train
MAE: 0.04217665446213075
RMSE: 0.11362884103431269
R²: 0.9540972706301667

HistGradientBoosting - Test
MAE: 0.07065365615845177
RMSE: 0.46329826804001195
R²: 0.7202341060300033


,Model,MAE,RMSE,R²
0,Random Forest - Train,0.015285,0.071600,0.981774
1,Random Forest - Test,0.062629,0.467523,0.715109
2,HistGradientBoosting - Train,0.042177,0.113629,0.954097
3,HistGradientBoosting - Test,0.070654,0.463298,0.720234


In [13]:
#a relancer en cas d'erreur
import importlib
import src.preparation
import src.model

importlib.reload(src.preparation)
importlib.reload(src.model)

<module 'src.model' from 'c:\\Users\\mathi\\OneDrive\\Documents\\Projet_IML\\src\\model.py'>

In [14]:
print("Train target:")
print(y_train.describe())

print("\nTest target:")
print(y_test.describe())

Train target:
count    8533.000000
mean        0.221976
std         0.530389
min         0.010000
25%         0.020000
50%         0.070000
75%         0.200000
max        12.880000
Name: EU_Sales, dtype: float64

Test target:
count    2134.000000
mean        0.240192
std         0.876123
min         0.010000
25%         0.020000
50%         0.070000
75%         0.200000
max        29.020000
Name: EU_Sales, dtype: float64


In [9]:
# Compare actual values with predictions
errors = pd.DataFrame({
    "Actual": y_test,
    "Predicted": rf_test_predictions
})

errors["Absolute_Error"] = abs(
    errors["Actual"] - errors["Predicted"]
)

# Display the 10 largest prediction errors
display(
    errors.sort_values(
        by="Absolute_Error",
        ascending=False
    ).head(10)
)

,Actual,Predicted,Absolute_Error
0,29.02,10.442877,18.577123
47,0.01,6.656049,6.646049
10,11.00,7.739543,3.260457
590,0.07,3.025041,2.955041
668,0.07,2.549341,2.479341
407,0.30,2.724197,2.424197
8,7.06,9.442123,2.382123
19,9.26,7.132300,2.127700
39,2.61,4.264588,1.654588
88,1.78,3.311220,1.531220


In [12]:
from sklearn.metrics import r2_score, mean_squared_error


# Remove the observation with the largest prediction error
errors = pd.DataFrame({
    "Actual": y_test,
    "Predicted": rf_test_predictions
})

errors["Absolute_Error"] = abs(
    errors["Actual"] - errors["Predicted"]
)

filtered_errors = errors.drop(
    errors["Absolute_Error"].idxmax()
)

# Recalculate metrics
new_rmse = np.sqrt(mean_squared_error(
    filtered_errors["Actual"],
    filtered_errors["Predicted"]
))

new_r2 = r2_score(
    filtered_errors["Actual"],
    filtered_errors["Predicted"]
)

print("New RMSE:", new_rmse)
print("New R²:", new_r2)

New RMSE: 0.23850481649597619
New R²: 0.8499459624434668
